# Model Comparison and ROC Curve Analysis

This notebook compares the performance of Logistic regression, Lasso logistic regression, random forest and xgboost machine learning models for multi-class classification of IBD (CD, UC, No IBD).
Pre-trained models are loaded and evaluated on a common test dataset for consistent comparison.

In [ ]:
import os
import joblib
import numpy as np
import sys

import warnings
warnings.filterwarnings('ignore')

plots_dir = "../../plots"
os.makedirs(plots_dir, exist_ok=True)

savedmodels_dir = "../../saved_models"
os.makedirs(savedmodels_dir, exist_ok=True)
sys.path.append(os.path.abspath(".."))

In [ ]:
from evaluation_plots import get_roc_data, plot_roc_comparison

## Evaluation Setup
Pre-trained models were loaded and evaluated on a common independent test dataset.
- All models use the same test data
- No retraining is performed in this notebook

In [ ]:
lr_model = joblib.load(f"{savedmodels_dir}/logisticregression_model.pkl")
rf_model = joblib.load(f"{savedmodels_dir}/Randomforest_model.pkl")
xgb_model = joblib.load(f"{savedmodels_dir}/XGBoost_model.pkl")

# RFECV components (since not pipeline)
lr_pre = joblib.load(f"{savedmodels_dir}/logisticregression_preprocessor.pkl")
lr_sel = joblib.load(f"{savedmodels_dir}/logisticregression_selector.pkl")
lr_rfecv = joblib.load(f"{savedmodels_dir}/logisticregression_rfecv_model.pkl")

rf_pre = joblib.load(f"{savedmodels_dir}/Randomforest_preprocessor.pkl")
rf_sel = joblib.load(f"{savedmodels_dir}/Randomforest_selector.pkl")
rf_rfecv = joblib.load(f"{savedmodels_dir}/Randomforest_rfecv_model.pkl")

xgb_pre = joblib.load(f"{savedmodels_dir}/XGBoost_preprocessor.pkl")
xgb_sel = joblib.load(f"{savedmodels_dir}/XGBoost_selector.pkl")
xgb_rfecv = joblib.load(f"{savedmodels_dir}/XGBoost_rfecv_model.pkl")

lasso_pre = joblib.load(f"{savedmodels_dir}/lassologisticregression_preprocessor.pkl")
lasso_sel = joblib.load(f"{savedmodels_dir}/lassologisticregression_selector.pkl")
lasso_model = joblib.load(f"{savedmodels_dir}/lassologisticregression_model.pkl")

# Common data
X_test = joblib.load(f"{savedmodels_dir}/X_test.pkl")
y_test = joblib.load(f"{savedmodels_dir}/y_test.pkl")
le = joblib.load(f"{savedmodels_dir}/label_encoder.pkl")

In [ ]:
y_prob_lr = lr_model.predict_proba(X_test)
y_prob_rf = rf_model.predict_proba(X_test)
y_prob_xgb = xgb_model.predict_proba(X_test)

In [ ]:
# LR
X_lr = lr_pre.transform(X_test)
X_lr = lr_sel.transform(X_lr)
y_prob_lr_rfecv = lr_rfecv.predict_proba(X_lr)
X_test.shape
# RF
X_rf = rf_pre.transform(X_test)
X_rf = rf_sel.transform(X_rf)
y_prob_rf_rfecv = rf_rfecv.predict_proba(X_rf)
X_test.shape
# XGB
X_xgb = xgb_pre.transform(X_test)
X_xgb = xgb_sel.transform(X_xgb)
y_prob_xgb_rfecv = xgb_rfecv.predict_proba(X_xgb)
X_test.shape

X_lasso = lasso_pre.transform(X_test)
X_lasso = lasso_sel.transform(X_lasso)
y_prob_lasso = lasso_model.predict_proba(X_lasso)
X_test.shape

In [ ]:
roc_lr = get_roc_data(y_test, y_prob_lr, le)
roc_rf = get_roc_data(y_test, y_prob_rf, le)
roc_xgb = get_roc_data(y_test, y_prob_xgb, le)

roc_lr_rfecv = get_roc_data(y_test, y_prob_lr_rfecv, le)
roc_rf_rfecv = get_roc_data(y_test, y_prob_rf_rfecv, le)
roc_xgb_rfecv = get_roc_data(y_test, y_prob_xgb_rfecv, le)
roc_lasso= get_roc_data(y_test, y_prob_lasso, le)

## ROC Curve Analysis

The ROC curves below compare model performance on the test dataset.

In [ ]:
roc_data_dict = {
    "Logistic Regression": roc_lr,
    "Random Forest": roc_rf,
    "XGBoost": roc_xgb
}
plot_roc_comparison(roc_data_dict, le, plots_dir)

In [ ]:
roc_dict_rfecv = {
    "Logistic Regression": roc_lr_rfecv,
    "Random Forest": roc_rf_rfecv,
    "XGBoost": roc_xgb_rfecv,
    "Lasso Logistic Regression": roc_lasso
}
plot_roc_comparison(roc_dict_rfecv, le, plots_dir, prefix="rfecv_")